# ISOM 835 · Session 11 — Forecasting: Time Series With ML & Foundation Models
**Suffolk University · Sawyer Business School · Fall 2026 · Mon Nov 30 · Prof. Hasan Arslan**

Why random splits lie when time matters, lag and rolling features that turn forecasting into regression, classical baselines you must beat, and zero-shot foundation models.

> **Frame the prediction (Bike Sharing).** *Unit:* one hour · *Target:* rentals · *Horizon:* 24 hours ahead · *Decision:* how many bikes to stage where · *Baseline:* same hour last week.

In [ ]:
import pandas as pd, numpy as np, io, zipfile, urllib.request
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

try:
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen('https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip', timeout=120).read()))
    bike = pd.read_csv(io.BytesIO(z.read('hour.csv')))
except Exception as e:
    print('download failed →', type(e).__name__, '— using synthetic hourly demand')
    idx = pd.date_range('2011-01-01', '2012-12-31 23:00', freq='h'); rng = np.random.default_rng(835)
    h = idx.hour; base = 60 + 180 * (np.exp(-((h - 8) ** 2) / 6) + np.exp(-((h - 17.5) ** 2) / 8)) * (idx.dayofweek < 5) + 120 * np.exp(-((h - 14) ** 2) / 30) * (idx.dayofweek >= 5)
    season = 1 + 0.6 * np.sin(2 * np.pi * (idx.dayofyear - 100) / 365); trend = np.linspace(0.7, 1.4, len(idx))
    bike = pd.DataFrame({'dteday': idx.strftime('%Y-%m-%d'), 'hr': h, 'holiday': 0, 'workingday': (idx.dayofweek < 5).astype(int), 'weathersit': rng.choice([1, 2, 3], len(idx), p=[.65, .25, .1]), 'temp': (0.5 + 0.35 * np.sin(2 * np.pi * (idx.dayofyear - 100) / 365) + rng.normal(0, .05, len(idx))).clip(0, 1), 'hum': rng.uniform(.3, .9, len(idx))})
    bike['cnt'] = (base * season * trend * np.where(bike['weathersit'] == 3, 0.5, 1) * rng.lognormal(0, 0.25, len(idx))).round().astype(int)
bike['ts'] = pd.to_datetime(bike['dteday']) + pd.to_timedelta(bike['hr'], unit='h')
bike = bike.set_index('ts').sort_index().asfreq('h')           # make missing hours explicit
print(bike.shape, f"missing hours filled: {bike['cnt'].isna().sum()}")
bike['cnt'] = bike['cnt'].interpolate()

## 1. The split that sees the future
A random split lets the model interpolate between neighboring hours. Only a time-ordered split is a forecast.

In [ ]:
y = bike['cnt']
cal = pd.DataFrame({'hour': bike.index.hour, 'dow': bike.index.dayofweek, 'month': bike.index.month, 'workingday': bike['workingday'].ffill(), 'holiday': bike['holiday'].ffill(),
                    'temp': bike['temp'].interpolate(), 'hum': bike['hum'].interpolate(), 'weathersit': bike['weathersit'].ffill()}, index=bike.index)
feat = cal.assign(lag_24=y.shift(24), lag_168=y.shift(168), roll_7d=y.shift(24).rolling(168).mean()).dropna(); y = y.loc[feat.index]
cut = feat.index.max() - pd.Timedelta(weeks=8)
X_tr, X_te, y_tr, y_te = feat[:cut], feat[cut:], y[:cut], y[cut:]

from sklearn.model_selection import train_test_split
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(feat, y, test_size=0.2, random_state=835)               # ❌ random
m_rand = HistGradientBoostingRegressor(random_state=835).fit(Xr_tr, yr_tr)
m_time = HistGradientBoostingRegressor(random_state=835).fit(X_tr, y_tr)                                # ✅ time-ordered
print(f'random split MAE  {mean_absolute_error(yr_te, m_rand.predict(Xr_te)):.1f}   (interpolation, not forecasting)')
print(f'time split MAE    {mean_absolute_error(y_te, m_time.predict(X_te)):.1f}   (the honest number)')

## 2. Baselines you must beat

In [ ]:
naive = y_te.shift(24).bfill(); seasonal = X_te['lag_168']; movavg = X_te['roll_7d']
for n, p in [('naive (24h ago)', naive), ('seasonal naive (168h ago)', seasonal), ('7-day moving average', movavg)]:
    print(f'{n:28s} MAE {mean_absolute_error(y_te, p):6.1f}   RMSE {root_mean_squared_error(y_te, p):6.1f}')

## 3. Classical models with StatsForecast
AutoETS, AutoARIMA, Theta on the hourly series — fast, well-tested, the right second baseline. No weather, no holidays: that is their limit.

In [ ]:
# OPTIONAL — pip install statsforecast
try:
    from statsforecast import StatsForecast
    from statsforecast.models import AutoETS, SeasonalNaive, AutoTheta
    train_sf = pd.DataFrame({'unique_id': 'bikes', 'ds': y_tr.index[-24*56:], 'y': y_tr.values[-24*56:]})      # last 8 weeks of training for speed
    sf = StatsForecast(models=[SeasonalNaive(season_length=168), AutoETS(season_length=24), AutoTheta(season_length=24)], freq='h')
    fc = sf.forecast(df=train_sf, h=24 * 7)                                                                        # one-week horizon
    truth = y_te.iloc[:24 * 7].values
    for col in ['SeasonalNaive', 'AutoETS', 'AutoTheta']:
        print(f'{col:14s} one-week MAE {mean_absolute_error(truth, fc[col].values):.1f}')
except ImportError:
    print('statsforecast not installed')

## 4. Forecasting as regression: lags + gradient boosting, validated in time order
Every lag respects the 24-hour horizon. `TimeSeriesSplit` keeps the validation folds after the training folds.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=24 * 14)
gbm = HistGradientBoostingRegressor(learning_rate=0.05, max_iter=2000, early_stopping=True, random_state=835)
cv_mae = -cross_val_score(gbm, X_tr, y_tr, cv=tscv, scoring='neg_mean_absolute_error')
print('rolling-origin CV MAE per fold:', cv_mae.round(1), '→ mean', cv_mae.mean().round(1))
gbm.fit(X_tr, y_tr); pred = gbm.predict(X_te)
print(f'test (last 8 weeks) MAE {mean_absolute_error(y_te, pred):.1f}   vs seasonal naive {mean_absolute_error(y_te, seasonal):.1f}   ({gbm.n_iter_} trees)')

In [ ]:
by_hour = pd.DataFrame({'gbm': (y_te - pred).abs(), 'seasonal': (y_te - seasonal).abs()}).groupby(y_te.index.hour).mean()
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
by_hour.plot(ax=ax[0], color=['#2ee6c5', '#ff6b8b'], title='MAE by hour of day'); ax[0].set_xlabel('hour')
wk = y_te.index[:24 * 7]; ax[1].plot(wk, y_te.loc[wk], color='#eef2f7', lw=1.2, label='actual'); ax[1].plot(wk, seasonal.loc[wk], color='#ff6b8b', lw=1, label='seasonal naive'); ax[1].plot(wk, pd.Series(pred, index=y_te.index).loc[wk], color='#2ee6c5', lw=1.4, label='GBM'); ax[1].legend(); ax[1].set_title('first test week')
plt.tight_layout(); plt.show()

## 5. Zero-shot: Chronos-2
A transformer pretrained on billions of time points forecasts your series with no training. Strong on periodic series, instant, Apache-2.0 — the new first baseline. (Runs on CPU in Colab; a GPU runtime is faster.)

In [ ]:
# OPTIONAL — Colab: !pip install -q chronos-forecasting   (downloads ~200MB of weights)
try:
    import torch
    from chronos import BaseChronosPipeline
    pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map='cpu', torch_dtype=torch.float32)
    context = torch.tensor(y_tr.values[-24 * 28:], dtype=torch.float32)                          # last 4 weeks as context
    q, mean = pipe.predict_quantiles(context=context, prediction_length=24 * 7, quantile_levels=[0.1, 0.5, 0.9])
    zs = mean[0].numpy(); truth = y_te.iloc[:24 * 7].values
    print(f'Chronos zero-shot one-week MAE {mean_absolute_error(truth, zs):.1f}   vs seasonal naive {mean_absolute_error(truth, seasonal.iloc[:24*7]):.1f}   vs GBM {mean_absolute_error(truth, pred[:24*7]):.1f}')
except ImportError:
    print('chronos-forecasting not installed — run the pip line above in Colab')

## 6. Your turn
1. **Three features of your own.** Add `lag_48`, a rolling max, and a "rain in the last 3 hours" flag. Does test MAE move?
2. **Holidays.** Compute MAE on holidays vs. non-holidays. Where would a DoorDash-style multiplier help?
3. **Horizon.** Change the task to 168 hours ahead (drop `lag_24`, `roll_7d` must shift by 168). How much worse is the honest number?

In [ ]:
# Your turn — work here

## What we learned tonight
- **Validate in time order.** Random splits let lag features see the future; `TimeSeriesSplit` and rolling-origin backtests are the honest scoreboard.
- **Forecasting is regression on features knowable at forecast time** — lags that respect the horizon, rolling stats, calendar, weather.
- **Beat seasonal naive, then StatsForecast, then run a foundation model.** Chronos/TimesFM are the instant baseline, not the finish line.

**HW5 Track B** due Dec 7. Project notebooks due Dec 13.